## 패리티 검사부호(짝수/홀수 패리티) 오류 검출 실습 프로그램.

### 사용 방법:
1) 스크립트를 실행한다.
2) 데이터 비트열(예: 1011001)을 입력한다.
3) 패리티 종류(짝수/홀수)를 선택한다.
4) 오류 주입 비트 위치를 입력하면(여러 개 가능), 수신 측에서 오류를 검출한다.

In [2]:
from __future__ import annotations


def validate_bits(bits: str) -> bool:
    return len(bits) > 0 and all(ch in {"0", "1"} for ch in bits)


def flip_bit(bit: str) -> str:
    return "1" if bit == "0" else "0"


def make_codeword(data_bits: str, mode: str) -> str:
    ones_count = data_bits.count("1")
    if mode == "even":
        parity_bit = "0" if ones_count % 2 == 0 else "1"
    else:  # odd
        parity_bit = "1" if ones_count % 2 == 0 else "0"
    return data_bits + parity_bit


def check_codeword(received_bits: str, mode: str) -> bool:
    ones_count = received_bits.count("1")
    if mode == "even":
        return ones_count % 2 == 0
    return ones_count % 2 == 1


def inject_errors(codeword: str, positions_1_based: list[int]) -> str:
    bits = list(codeword)
    for pos in positions_1_based:
        idx = pos - 1
        bits[idx] = flip_bit(bits[idx])
    return "".join(bits)


def parse_error_positions(raw: str, max_len: int) -> list[int]:
    if not raw.strip():
        return []

    tokens = raw.replace(",", " ").split()
    positions: list[int] = []
    for t in tokens:
        if not t.isdigit():
            raise ValueError(f"숫자가 아닌 입력: {t}")
        value = int(t)
        if value < 1 or value > max_len:
            raise ValueError(f"범위를 벗어난 위치: {value} (허용: 1~{max_len})")
        positions.append(value)
    return positions


def mode_text(mode: str) -> str:
    return "짝수 패리티" if mode == "even" else "홀수 패리티"


def run_interactive() -> None:
    print("=" * 60)
    print("패리티 검사부호 오류 검출 실습")
    print("=" * 60)

    while True:
        data_bits = input("\n데이터 비트열 입력(0/1): ").strip()
        if validate_bits(data_bits):
            break
        print("입력 오류: 0과 1로만 구성된 비트열을 입력하세요.")

    while True:
        mode_in = input("패리티 선택 [E]ven(짝수) / [O]dd(홀수): ").strip().lower()
        if mode_in in {"e", "even"}:
            mode = "even"
            break
        if mode_in in {"o", "odd"}:
            mode = "odd"
            break
        print("입력 오류: E(짝수) 또는 O(홀수)를 입력하세요.")

    codeword = make_codeword(data_bits, mode)
    print("\n[송신 측]")
    print(f"- 선택한 방식: {mode_text(mode)}")
    print(f"- 원본 데이터 : {data_bits}")
    print(f"- 생성 코드워드(데이터+패리티): {codeword}")
    print(f"  (마지막 비트가 패리티 비트)")

    print("\n[전송 중 오류 주입]")
    print(f"- 위치는 1부터 시작, 범위는 1~{len(codeword)}")
    print("- 예: 3 또는 2,5 또는 1 4 7")
    print("- 엔터만 누르면 오류 없이 전송")

    while True:
        raw = input("비트 반전 위치 입력: ").strip()
        try:
            error_positions = parse_error_positions(raw, len(codeword))
            break
        except ValueError as ex:
            print(f"입력 오류: {ex}")

    received = inject_errors(codeword, error_positions)

    print("\n[수신 측]")
    print(f"- 수신 코드워드: {received}")
    is_ok = check_codeword(received, mode)
    if is_ok:
        print("- 패리티 검사 결과: 정상(오류 미검출)")
    else:
        print("- 패리티 검사 결과: 오류 검출!")

    if error_positions:
        print(f"\n참고: 실제 주입된 오류 위치 = {error_positions}")
        if len(error_positions) % 2 == 0:
            print("주의: 패리티 검사는 짝수 개 비트 오류가 발생하면 검출에 실패할 수 있습니다.")
    else:
        print("\n참고: 오류를 주입하지 않았습니다.")


if __name__ == "__main__":
    run_interactive()


패리티 검사부호 오류 검출 실습



데이터 비트열 입력(0/1):  1110001001010010101
패리티 선택 [E]ven(짝수) / [O]dd(홀수):  e



[송신 측]
- 선택한 방식: 짝수 패리티
- 원본 데이터 : 1110001001010010101
- 생성 코드워드(데이터+패리티): 11100010010100101011
  (마지막 비트가 패리티 비트)

[전송 중 오류 주입]
- 위치는 1부터 시작, 범위는 1~20
- 예: 3 또는 2,5 또는 1 4 7
- 엔터만 누르면 오류 없이 전송


비트 반전 위치 입력:  2,4,6



[수신 측]
- 수신 코드워드: 10110110010100101011
- 패리티 검사 결과: 오류 검출!

참고: 실제 주입된 오류 위치 = [2, 4, 6]
